# Day 2 — API Response Anatomy + Chatbot with Memory
---
> **Phase:** 1 — LLM Foundations  
> **Status:** ✅ Complete

## 🧠 Key Concepts

### 1. API Response Object Anatomy

The API returns a `ChatCompletion` object — not just text. It contains:

```
ChatCompletion
├── id                          → unique request ID
├── model                       → model used
├── choices[]                   → LIST of completions (usually 1)
│   └── choices[0]
│       ├── finish_reason       → why generation stopped
│       │   ├── 'stop'          → finished naturally
│       │   ├── 'length'        → hit max_tokens limit
│       │   └── 'tool_calls'    → model called a tool
│       ├── message
│       │   ├── role            → 'assistant'
│       │   ├── content         → the actual text response
│       │   └── tool_calls      → None (unless agent called a tool)
│       └── logprobs            → token probabilities (if requested)
└── usage
    ├── prompt_tokens           → tokens you sent
    ├── completion_tokens       → tokens model generated
    └── total_tokens            → prompt + completion
```

**Why `choices` is a list:**
- Parameter `n` lets you request multiple completions in one call
- Default `n=1` → always use `choices[0]`
- `n=3` → model returns 3 different responses → `choices[0]`, `choices[1]`, `choices[2]`

**Why token tracking matters:**
- Every provider charges per token
- Input and output tokens are priced differently (output usually costs more)
- In production with thousands of users → token bloat = hundreds of dollars extra

### 2. Error Types You'll See Constantly

| Error Code | Meaning | Your Fault? |
|-----------|---------|-------------|
| 400 | Bad request — wrong parameters | Yes |
| 401 | Authentication error — wrong API key | Yes |
| 429 | Rate limit / quota exceeded | Sometimes |
| 500 | Server error | Provider's fault |

### 3. How Chatbot Memory Actually Works

**The model has NO memory between calls. Every call is stateless.**

Memory lives in YOUR code — as a list of dictionaries:

```python
conversation_history = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "My name is Sara."},
    {"role": "assistant", "content": "Hello Sara! How can I help?"},
    {"role": "user",      "content": "What is a token?"},
    {"role": "assistant", "content": "A token is..."},
    {"role": "user",      "content": "What did I tell you my name was?"}
    # ↑ model sees ALL of the above and knows the answer is Sara
]
```

**The pattern every turn:**
1. Append user message to history
2. Send ENTIRE history to API
3. Get response
4. Append assistant response to history
5. Repeat

### 4. Memory Management Strategies

**Problem:** Long conversations → too many tokens → cost bloat + context overflow

| Strategy | How | Tradeoff |
|---------|-----|----------|
| Sliding Window | Keep only last N messages | Cheap, simple, forgets old context |
| Summarization | Compress old messages into summary | Moderate cost, retains compressed history |
| Relevant Retrieval | Fetch only history relevant to current question | Foundation of RAG |
| Hybrid | Summarize old + retrieve relevant + keep recent full | Most sophisticated, production-grade |

**Sliding Window Implementation:**
```python
# Always keep system message + last 6 messages
messages = [conversation_history[0]] + conversation_history[-6:]
```

**Why `[conversation_history[0]]` and not `conversation_history[0]`:**
- `conversation_history[0]` = a dictionary
- `conversation_history[-6:]` = a list
- Can't use `+` between dict and list
- `[conversation_history[0]]` = wraps dict in a list → both sides are lists → `+` works

## 💻 Code

In [ ]:
# Day 2 — Extracting Response Object Fields
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is a token in LLMs? Answer in 2 sentences."}
    ],
    temperature=0.7,
    max_tokens=500
)

# Extracting individual fields
print("Response ID:       ", response.id)
print("Model used:        ", response.model)
print("Finish reason:     ", response.choices[0].finish_reason)
print("Assistant reply:   ", response.choices[0].message.content)
print("Role:              ", response.choices[0].message.role)
print("Prompt tokens:     ", response.usage.prompt_tokens)
print("Completion tokens: ", response.usage.completion_tokens)
print("Total tokens:      ", response.usage.total_tokens)

In [ ]:
# Day 2 — Chatbot with Sliding Window Memory
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# This list IS the chatbot's memory
conversation_history = [
    {"role": "system", "content": "You are a helpful assistant."}
]

print("Chatbot ready. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")
    
    if user_input.lower() == "quit":
        break
    
    # Step 1: Append user message
    conversation_history.append({"role": "user", "content": user_input})
    
    # Step 2: Send with sliding window (system message + last 6)
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[conversation_history[0]] + conversation_history[-6:],
        temperature=0.7,
        max_tokens=500
    )
    
    # Step 3: Extract reply
    assistant_reply = response.choices[0].message.content
    
    # Step 4: Append assistant reply
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    print(f"\nBot: {assistant_reply}")
    print(f"Tokens used this call: {response.usage.completion_tokens}")
    print(f"Total conversation tokens so far: {response.usage.total_tokens}\n")

## ✅ Day 2 Summary

```
✓ Full anatomy of API response object
✓ Why choices is a list (n parameter)
✓ How token usage tracking works
✓ finish_reason values: stop, length, tool_calls
✓ How chatbot memory works (list of dicts)
✓ Why memory management is a real problem
✓ Sliding window memory — implemented from scratch
✓ Four production memory strategies
✓ Token cost tracking per call
```